To use this notebook you first have to follow these steps:

1. Run the following command in your shell from the root directory of the repo: ```./init.sh ``` 
    - This command will install all the dependencies and initialize the submodules (it was written and tested on Ubuntu 20.04 running on AWS g6 ec2 instances)
2. Login to Huggingface and Weights & Biases from your CLI:
    - ```huggingface-cli login [Auth Token]```
    - ```wandb login [Auth Token]```
3. Update the Benchmark Config Files at 
    - ```llm_judge/arena-hard-auto/config/api_config.yaml```
    - ```llm_judge/arena-hard-auto/config/gen_answer_config.yaml```
    - ```llm_judge/arena-hard-auto/config/judge_config.yaml``` 

4. Start your LLM on an OpenAI API Server with vLLM using one of the following commands: 
    - With Docker: 

    ```docker run --runtime nvidia --gpus all -v ~/.cache/huggingface:/root/.cache/huggingface -p 8000:8000 --ipc=host vllm/vllm-openai:latest --model [model name (huggingface model id)] --tensor-parallel-size [number of gpus] --max-model-len 8192 --gpu-memory-utilization 0.85 --enable-chunked-prefill --served-model-name [api name for the model]```

    
    - Without Docker: 

    ```python -m vllm.entrypoints.openai.api_server --model [model name (huggingface model id)] --tensor-parallel-size [number of gpus] --max-model-len 8192 --gpu-memory-utilization 0.9 --enable-chunked-prefill --served-model-name [api name for the model]```


After these steps you should see the message that your model is running on the address ```http://0.0.0.0:8000```





Additionally:
Running the model on CPU without GPU: 

1. Build the Docker image. Run the following command from within the vllm submodule folder: 
- ```docker build -f Dockerfile.cpu -t vllm-cpu-env --shm-size=4g .```

2. Run the Docker Container: 
- ```docker run -it --rm --network=host -v ~/.cache/huggingface:/root/.cache/huggingface -p 8000:8000 vllm-cpu-env --model [model name (huggingface model id)] --max-model-len 8192 --enable-chunked-prefill --served-model-name [api name for the model]```

# Optimal setups for different model and instance sizes

Setup for llama3.1-70B-FP8 on a 8 GPU Instance (g6.48xl): 

```docker run --runtime nvidia --gpus all -v ~/.cache/huggingface:/root/.cache/huggingface -p 8000:8000 --ipc=host vllm/vllm-openai:latest --model neuralmagic/Meta-Llama-3.1-70B-Instruct-FP8 --tensor-parallel-size 8 --max-model-len 8192 --gpu-memory-utilization 0.80 --enable-chunked-prefill --served-model-name llama3_1_70b_fp8```


Setup for llama3.1-70B-AWQ-INT4 on 8 GPUs (NOT OPTIMAL YET)

```docker run --runtime nvidia --gpus all -v ~/.cache/huggingface:/root/.cache/huggingface -p 8000:8000 --ipc=host vllm/vllm-openai:latest --model hugging-quants/Meta-Llama-3.1-70B-Instruct-AWQ-INT4 --tensor-parallel-size 8 --max-model-len 8192 --gpu-memory-utilization 0.85 --enable-chunked-prefill --served-model-name llama3_1_70b_awq_int4 --tokenizer-pool-size 32```


# LOOK INTO

--tokenizer-pool-size
Size of tokenizer pool to use for asynchronous tokenization. If 0, will use synchronous tokenization.

Default: 0

--pipeline-parallel-size, -pp
Number of pipeline stages.

Default: 1



# Setup

In [1]:
import torch
import pandas as pd
from transformers import  AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, pipeline
import os
from codecarbon import EmissionsTracker
import time
import csv
import json
import yaml
from openai import OpenAI

import tiktoken

import wandb

import random

In [2]:
# Get the number of available GPUs
num_gpus = torch.cuda.device_count()

print(f"Number of available GPUs: {num_gpus}")

Number of available GPUs: 4


# Settings up the configs

### Answer Config and Benchmark Details

In [3]:
runs = 3

#We have to select the same tokenizer all the time in order to get the same Token numbers at the end (this is only used to calculate the number of tokens)
huggingface_model_name = "meta-llama/Meta-Llama-3.1-8B-Instruct"

model_name = "llama3_1_8b_fp8"
bench_name = 'arena-hard-v0.1'
max_gen_length = 4096 # Questions in the arena hard auto benchmark are sometimes longer than 2048 tokens. Therefore the model length has to be longer than 4096 tokens.
temperature = 0.0
num_choices = 1

config_filename = 'arena-hard-auto/config/answer_config_temp.yaml'
question_path = f'arena-hard-auto/data/{bench_name}/question.jsonl'
guidance_path = f'arena-hard-auto/data/{bench_name}/guidance.jsonl'

print("Config will be written to:\n  " + config_filename)

Config will be written to:
  arena-hard-auto/config/answer_config_temp.yaml


In [4]:
# Set the default start run (can be modified)
startat = 1  # Default to start at run_1

# Ensure startat is within the valid range
if startat < 1 or startat > runs:
    raise ValueError(f"startat should be between 1 and {runs}")


print(f"Starting at run: {startat} out of {runs}")


Starting at run: 1 out of 3


### Endpoint Config

In [5]:
api_base = 'http://localhost:8000/v1'
api_key = 'EMPTY'
api_type = 'openai'
parallel = 50
system_prompt = 'cluster_info'


endpoint_filename = 'arena-hard-auto/config/api_config_temp.yaml'

print("Endpoint Config will be written to:\n  " + endpoint_filename)

Endpoint Config will be written to:
  arena-hard-auto/config/api_config_temp.yaml


Setup Tokenizer to Count Tokens

In [6]:
# Load the tokenizer
tokenizer = AutoTokenizer.from_pretrained(huggingface_model_name, padding_side="left")
tokenizer.pad_token = tokenizer.bos_token

# Run the Benchmark with Energy Consumption Tracking

In [7]:
def get_questions(question_file: str):
    """Load questions from a file into a list."""

    questions = []
    with open(question_file, "r") as ques_file:
        for line in ques_file:
            if line:
                questions.append(json.loads(line))

    return questions

In [8]:
def get_answers(answer_file: str):
    """Load model answers."""

    answers = []
    with open(answer_file, "r") as ans_file:
        for line in ans_file:
            if line:
                answers.append(json.loads(line))

    return answers


In [9]:
def load_guidance(guidance_file: str):
    """Load guidance from a file."""
    guidance = {}
    with open(guidance_file, "r") as fin:
        for line in fin:
            if line:
                line = json.loads(line)
                guidance[line["question_id"]] = line
    return guidance

## Runs without Guidance

In [10]:
print("="*10 + f" Starting Benchmark {bench_name} with {model_name} from run {startat}" + "="*10 + "\n\n")

question_path = f"arena-hard-auto/data/{bench_name}/question.jsonl"

questions = get_questions(question_path)

# Adjust the loop to start at 'startat' and go up to 'runs'
for run in range(startat - 1, runs):
    # Start non Guided Runs
    print("-"*20 + f"Starting Run {run+1}/{runs} (not guided)" + "-"*20)

    answer_dir_name = f'arena-hard-auto/data/{bench_name}/batched_model_answer/{model_name}'
    answer_path_arg = f"batched_model_answer/{model_name}"

    model_alias = f"{model_name}_run_{run+1}"
    answer_filename = os.path.join(answer_dir_name, f"{model_alias}.jsonl")

    name = f"{model_name}-{bench_name}-{num_gpus}gpus-run_{run+1}"

    add_guidance = False
    guidance_only = False

    # Define the data to be written to the YAML file
    config_data = {
        'name': name,
        'bench_name': bench_name,
        'temperature': temperature,
        'max_tokens': max_gen_length,
        'num_choices': num_choices,
        'add_guidance': add_guidance,
        'guidance_only': guidance_only,
        'answer_path': answer_path_arg,
        'model_list': [
            model_alias
        ]
    }

    # Define the data to be written to the YAML file
    endpoint_config = {
        model_alias: {
            'model_name': model_name,
            'endpoints': [
                {
                    'api_base': api_base,
                    'api_key': api_key,
                }
            ],
            'api_type': api_type,
            'parallel': parallel,
            'system_prompt': system_prompt
        }
    }

    # Delete temporary config file if it exists
    if os.path.exists(config_filename):
        os.remove(config_filename)
        print(f"OLD Configuration file '{config_filename}' deleted successfully.")
    else:
        print(f"No OLD Configuration file found at '{config_filename}'")

    if os.path.exists(endpoint_filename):
        os.remove(endpoint_filename)
        print(f"OLD Endpoint Config file '{endpoint_filename}' deleted successfully.")
    else:
        print(f"No OLD Endpoint Config file found at '{endpoint_filename}'")

    if os.path.exists(answer_filename):
        os.remove(answer_filename)
        print(f"OLD Bench results file '{answer_filename}' deleted successfully.")
    else:
        print(f"No OLD Bench results file found at '{answer_filename}'")


    # Write the config to a temporary YAML file
    with open(config_filename, 'w') as file:
        yaml.dump(config_data, file, default_flow_style=False)

    with open(endpoint_filename, 'w') as file:
        yaml.dump(endpoint_config, file, default_flow_style=False)

    print(f"Configuration file '{config_filename}' created successfully.")
    print(f"Endpoint Config file '{endpoint_filename}' created successfully.")


    prompts = []

    for question in questions:

        system_prompt = f"""
            You are a sophisticated AI-Expert there to help users solve tasks in several domains efficiently and accurately.
            Now solve the following task from the domain "{question['cluster']}".\n
            """
        
        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": question["turns"][0]["content"]},
        ]
        prompt = system_prompt + question["turns"][0]["content"]

        prompts.append(prompt)

    num_prompts = len(prompts)

    total_input_tok = 0
    total_output_tok = 0


    wandb.init(
        # set the wandb project where this run will be logged
        project="Model_Benchmarks",

        # track hyperparameters and run metadata
        config={
        "benchmark_name": bench_name,
        "num_prompts": num_prompts,
        "framework": 'vLLM',
        "model": model_name,
        "num_gpus": num_gpus,
        },

        name=name,
    )


    #### Start The Benchmark

    tracker = EmissionsTracker(save_to_file=True, project_name=f"{name}", log_level="error", pue = 1.22, output_file=f"emissions_batched_benchmakrs.csv")
    tracker.start()

    # Start Timer for Inference
    start_time = time.time()

    # Change the current working directory to 'arena-hard-auto'
    os.chdir('arena-hard-auto')

    # Run the benchmark
    %run -i 'gen_answer.py' --setting-file config/answer_config_temp.yaml --endpoint-file config/api_config_temp.yaml --no-confirmation

    # End Timer for Inference
    end_time = time.time()

    os.chdir('..')


    emissions: float = tracker.stop()

    ttime = end_time-start_time


    print(f"\n\nFinished Benchmark in {ttime:.2f}s")


    #### End the Benchmark


    answers_file = get_answers(answer_filename)
    outputs = [answer["choices"][0]["turns"][0]["content"] for answer in answers_file]


    for idx, output in enumerate(outputs): 

        # Extracting information
        prompt = prompts[idx]
        input_tokens = tokenizer.encode(prompt)
        output_tokens = tokenizer.encode(output)
        num_input_tokens = len(input_tokens)
        num_output_tokens = len(output_tokens)

        # Updating cumulative counts
        total_input_tok += num_input_tokens
        total_output_tok += num_output_tokens


    # Calculate averages
    avg_time_per_prompt = (ttime / num_prompts)
    avg_toks_per_sec = total_output_tok/ttime
    avg_input_tokens = total_input_tok / num_prompts
    avg_output_tokens = total_output_tok / num_prompts

    em_i = emissions/total_input_tok *1_000_000
    em_o = emissions/total_output_tok *1_000_000
    em_p = emissions/num_prompts *10_000

    print("="*15 + f" RESULTS for {name} " + "="*15 + 
        "\n\n" + 
        f"""
        Finished Benchmark {bench_name} with {model_name}\n\n
        Total Time: {ttime:.2f}s, AVG/Prompt: {avg_time_per_prompt:.2f}s\n\n
        Average tokens per second: {avg_toks_per_sec:.2f}\n\n
        Total Prompts: {num_prompts}\n
        Total Input Tokens: {total_input_tok}, AVG/Prompt: {avg_input_tokens}\n
        Total Output Tokens: {total_output_tok}, AVG/Prompt: {avg_output_tokens}\n
        """ + 
        
        "-"*50 + "\n"
        )

    wandb.log({"Total Time": ttime,
        "AVG. Time / Prompt": avg_time_per_prompt,
                "AVG. Tokens / Second": avg_toks_per_sec,
                "AVG. Input Tokens": avg_input_tokens,
                "AVG. Output Tokens": avg_output_tokens,
                "Total Emissions": emissions,
                "Emissions / 1.000.000 Input Tokens": em_i,
                "Emissions / 1.000.000 Output Tokens": em_o,
                "Emissions / 10.000 Prompts": em_p,
                })

    wandb.finish()

    # Save results to a CSV file
    results = [
        ["Model", model_name],
        ["Benchmark", bench_name],
        ["Number of GPUs", num_gpus],
        ["Total Prompts", num_prompts],
        ["Total Time", ttime], 
        ["AVG. Time / Prompt", avg_time_per_prompt],
        ["AVG. Tokens / Second", avg_toks_per_sec],
        ["Total Input Tokens", total_input_tok],
        ["AVG. Input Tokens / Prompt", avg_input_tokens],
        ["Total Output Tokens", total_output_tok],
        ["AVG. Output Tokens / Prompt", avg_output_tokens],
        ["Total Emissions", emissions],
        ["Emissions / 1.000.000 Input Tokens", em_i],
        ["Emissions / 1.000.000 Output Tokens", em_o],
        ["Emissions / 10.000 Prompts", em_p]
    ]

    # Ensure the directory exists
    emission_output_file_path = f"emission_data/{name}_emission_data.csv"
    os.makedirs(os.path.dirname(emission_output_file_path), exist_ok=True)

    with open(emission_output_file_path, 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(["Metric", "Value"])
        writer.writerows(results)

    print(f"Results saved to {emission_output_file_path}\n\n")

Failed to detect the name of this notebook, you can set it manually with the WANDB_NOTEBOOK_NAME environment variable to enable code saving.


========== Starting Benchmark arena-hard-v0.1 with llama3_1_8b_fp8 from run 1==========


--------------------Starting Run 1/3 (not guided)--------------------
OLD Configuration file 'arena-hard-auto/config/answer_config_temp.yaml' deleted successfully.
OLD Endpoint Config file 'arena-hard-auto/config/api_config_temp.yaml' deleted successfully.
No OLD Bench results file found at 'arena-hard-auto/data/arena-hard-v0.1/batched_model_answer/llama3_1_8b_fp8/llama3_1_8b_fp8_run_1.jsonl'
Configuration file 'arena-hard-auto/config/answer_config_temp.yaml' created successfully.
Endpoint Config file 'arena-hard-auto/config/api_config_temp.yaml' created successfully.


wandb: Currently logged in as: daniel-wetzel (llm-emissions). Use `wandb login --relogin` to force relogin


ref: /opt/conda/envs/pytorch/lib/python3.11/site-packages/codecarbon/data/hardware/cpu_power.csv
{'add_guidance': False, 'answer_path': 'batched_model_answer/llama3_1_8b_fp8', 'bench_name': 'arena-hard-v0.1', 'guidance_only': False, 'max_tokens': 4096, 'model_list': ['llama3_1_8b_fp8_run_1'], 'name': 'llama3_1_8b_fp8-arena-hard-v0.1-4gpus-run_1', 'num_choices': 1, 'temperature': 0.0}
=========================  Expected Costs (based on GPT-4o)  =========================

Expected Input Tokens: 
 47461 Tokens in a total of 500 questions

Expected Output Tokens: 
 275000 Tokens in a total of 500 questions

Max Output Tokens: 
 400000 Tokens in a total of 500 questions


-------------------------  Resulting in Costs:   -------------------------

Expected Costs: 
 4.36 USD

Max. Expected Costs: 
 6.24 USD

Starting to generate answers...
Output to data/arena-hard-v0.1/batched_model_answer/llama3_1_8b_fp8/llama3_1_8b_fp8_run_1.jsonl


 37%|███▋      | 183/500 [01:57<03:41,  1.43it/s]

...

ref: /opt/conda/envs/pytorch/lib/python3.11/site-packages/codecarbon/data/private_infra/2016/usa_emissions.json


100%|██████████| 500/500 [05:25<00:00,  1.54it/s]


ref: /opt/conda/envs/pytorch/lib/python3.11/site-packages/codecarbon/data/private_infra/2016/usa_emissions.json


Finished Benchmark in 326.10s
=============== RESULTS for llama3_1_8b_fp8-arena-hard-v0.1-4gpus-run_3 ===============


        Finished Benchmark arena-hard-v0.1 with llama3_1_8b_fp8


        Total Time: 326.10s, AVG/Prompt: 0.65s


        Average tokens per second: 995.57


        Total Prompts: 500

        Total Input Tokens: 67571, AVG/Prompt: 135.142

        Total Output Tokens: 324656, AVG/Prompt: 649.312

        --------------------------------------------------



AVG. Input Tokens,▁
AVG. Output Tokens,▁
AVG. Time / Prompt,▁
AVG. Tokens / Second,▁
Emissions / 1.000.000 Input Tokens,▁
Emissions / 1.000.000 Output Tokens,▁
Emissions / 10.000 Prompts,▁
Total Emissions,▁
Total Time,▁
AVG. Input Tokens,135.142
AVG. Output Tokens,649.312


Results saved to emission_data/llama3_1_8b_fp8-arena-hard-v0.1-4gpus-run_3_emission_data.csv




## Runs with Guidance

In [11]:
print("="*10 + f" Starting Guided Benchmark {bench_name} with {model_name} from run {startat}" + "="*10 + "\n\n")

question_path = f"arena-hard-auto/data/{bench_name}/question.jsonl"

questions = get_questions(question_path)
guidances = load_guidance(guidance_path)

# Adjust the loop to start at 'startat' and go up to 'runs'
for run in range(startat - 1, runs):
    # Start Guided Runs
    print("-"*20 + f"Starting Run {run+1}/{runs} (guided)" + "-"*20)

    answer_dir_name = f'arena-hard-auto/data/{bench_name}/batched_model_answer/{model_name}_guided'
    answer_path_arg = f"batched_model_answer/{model_name}_guided"

    model_alias = f"{model_name}_guided_run_{run+1}"
    answer_filename = os.path.join(answer_dir_name, f"{model_alias}_guided.jsonl")

    name = f"{model_name}_guided-{bench_name}-{num_gpus}gpus-run_{run+1}"

    add_guidance = True
    guidance_only = True

    # Define the data to be written to the YAML file
    config_data = {
        'name': name,
        'bench_name': bench_name,
        'temperature': temperature,
        'max_tokens': max_gen_length,
        'num_choices': num_choices,
        'add_guidance': add_guidance,
        'guidance_only': guidance_only,
        'answer_path': answer_path_arg,
        'model_list': [
            model_alias
        ]
    }

    # Define the data to be written to the YAML file
    endpoint_config = {
        model_alias: {
            'model_name': model_name,
            'endpoints': [
                {
                    'api_base': api_base,
                    'api_key': api_key,
                }
            ],
            'api_type': api_type,
            'parallel': parallel,
            'system_prompt': system_prompt
        }
    }

    # Delete temporary config file if it exists
    if os.path.exists(config_filename):
        os.remove(config_filename)
        print(f"OLD Configuration file '{config_filename}' deleted successfully.")
    else:
        print(f"No OLD Configuration file found at '{config_filename}'")

    if os.path.exists(endpoint_filename):
        os.remove(endpoint_filename)
        print(f"OLD Endpoint Config file '{endpoint_filename}' deleted successfully.")
    else:
        print(f"No OLD Endpoint Config file found at '{endpoint_filename}'")

    if os.path.exists(answer_filename):
        os.remove(answer_filename)
        print(f"OLD Bench results file '{answer_filename}' deleted successfully.")
    else:
        print(f"No OLD Bench results file found at '{answer_filename}'")


    # Write the config to a temporary YAML file
    with open(config_filename, 'w') as file:
        yaml.dump(config_data, file, default_flow_style=False)

    with open(endpoint_filename, 'w') as file:
        yaml.dump(endpoint_config, file, default_flow_style=False)

    print(f"Configuration file '{config_filename}' created successfully.")
    print(f"Endpoint Config file '{endpoint_filename}' created successfully.")


    prompts = []

    for question in questions:

        guidance = guidances.get(question["question_id"], "")

        system_prompt = f"""
            You are a sophisticated AI assistant with the task to solve a question in the domain of {question['cluster']}. 
            Here is important guidance to solve the task that will be given to you:\n{guidance}\n\n
            Your task is:\n"""
        
        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": question["turns"][0]["content"]},
        ]
        prompt = system_prompt + question["turns"][0]["content"]

        prompts.append(prompt)

    num_prompts = len(prompts)

    total_input_tok = 0
    total_output_tok = 0


    wandb.init(
        # set the wandb project where this run will be logged
        project="Model_Benchmarks",

        # track hyperparameters and run metadata
        config={
        "benchmark_name": bench_name,
        "num_prompts": num_prompts,
        "framework": 'vLLM',
        "model": f"{model_name}_guided",
        "num_gpus": num_gpus,
        },

        name=name,
    )


    #### Start The Benchmark

    tracker = EmissionsTracker(save_to_file=True, project_name=f"{name}", log_level="error", pue = 1.22, output_file=f"emissions_batched_benchmakrs.csv")
    tracker.start()

    # Start Timer for Inference
    start_time = time.time()

    # Change the current working directory to 'arena-hard-auto'
    os.chdir('arena-hard-auto')

    # Run the benchmark
    %run -i 'gen_answer.py' --setting-file config/answer_config_temp.yaml --endpoint-file config/api_config_temp.yaml --no-confirmation

    # End Timer for Inference
    end_time = time.time()

    os.chdir('..')


    emissions: float = tracker.stop()

    ttime = end_time-start_time


    print(f"\n\nFinished Benchmark in {ttime:.2f}s")


    #### End the Benchmark


    answers_file = get_answers(answer_filename)
    outputs = [answer["choices"][0]["turns"][0]["content"] for answer in answers_file]


    for idx, output in enumerate(outputs): 

        # Extracting information
        prompt = prompts[idx]
        input_tokens = tokenizer.encode(prompt)
        output_tokens = tokenizer.encode(output)
        num_input_tokens = len(input_tokens)
        num_output_tokens = len(output_tokens)

        # Updating cumulative counts
        total_input_tok += num_input_tokens
        total_output_tok += num_output_tokens


    # Calculate averages
    avg_time_per_prompt = (ttime / num_prompts)
    avg_toks_per_sec = total_output_tok/ttime
    avg_input_tokens = total_input_tok / num_prompts
    avg_output_tokens = total_output_tok / num_prompts

    em_i = emissions/total_input_tok *1_000_000
    em_o = emissions/total_output_tok *1_000_000
    em_p = emissions/num_prompts *10_000

    print("="*15 + f" RESULTS for {name} " + "="*15 + 
        "\n\n" + 
        f"""
        Finished Benchmark {bench_name} with {model_name}\n\n
        Total Time: {ttime:.2f}s, AVG/Prompt: {avg_time_per_prompt:.2f}s\n\n
        Average tokens per second: {avg_toks_per_sec:.2f}\n\n
        Total Prompts: {num_prompts}\n
        Total Input Tokens: {total_input_tok}, AVG/Prompt: {avg_input_tokens}\n
        Total Output Tokens: {total_output_tok}, AVG/Prompt: {avg_output_tokens}\n
        """ + 
        
        "-"*50 + "\n"
        )

    wandb.log({"Total Time": ttime,
        "AVG. Time / Prompt": avg_time_per_prompt,
                "AVG. Tokens / Second": avg_toks_per_sec,
                "AVG. Input Tokens": avg_input_tokens,
                "AVG. Output Tokens": avg_output_tokens,
                "Total Emissions": emissions,
                "Emissions / 1.000.000 Input Tokens": em_i,
                "Emissions / 1.000.000 Output Tokens": em_o,
                "Emissions / 10.000 Prompts": em_p,
                })

    wandb.finish()

    # Save results to a CSV file
    results = [
        ["Model", model_name],
        ["Benchmark", bench_name],
        ["Number of GPUs", num_gpus],
        ["Total Prompts", num_prompts],
        ["Total Time", ttime], 
        ["AVG. Time / Prompt", avg_time_per_prompt],
        ["AVG. Tokens / Second", avg_toks_per_sec],
        ["Total Input Tokens", total_input_tok],
        ["AVG. Input Tokens / Prompt", avg_input_tokens],
        ["Total Output Tokens", total_output_tok],
        ["AVG. Output Tokens / Prompt", avg_output_tokens],
        ["Total Emissions", emissions],
        ["Emissions / 1.000.000 Input Tokens", em_i],
        ["Emissions / 1.000.000 Output Tokens", em_o],
        ["Emissions / 10.000 Prompts", em_p]
    ]

    # Ensure the directory exists
    emission_output_file_path = f"emission_data/{name}_emission_data.csv"
    os.makedirs(os.path.dirname(emission_output_file_path), exist_ok=True)

    with open(emission_output_file_path, 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(["Metric", "Value"])
        writer.writerows(results)

    print(f"Results saved to {emission_output_file_path}\n\n")

========== Starting Guided Benchmark arena-hard-v0.1 with llama3_1_8b_fp8 from run 1==========


--------------------Starting Run 1/1 (guided)--------------------
OLD Configuration file 'arena-hard-auto/config/answer_config_temp.yaml' deleted successfully.
OLD Endpoint Config file 'arena-hard-auto/config/api_config_temp.yaml' deleted successfully.
No OLD Bench results file found at 'arena-hard-auto/data/arena-hard-v0.1/batched_model_answer/llama3_1_8b_fp8_guided/llama3_1_8b_fp8_guided_run_1_guided.jsonl'
Configuration file 'arena-hard-auto/config/answer_config_temp.yaml' created successfully.
Endpoint Config file 'arena-hard-auto/config/api_config_temp.yaml' created successfully.


ref: /opt/conda/envs/pytorch/lib/python3.11/site-packages/codecarbon/data/hardware/cpu_power.csv
{'add_guidance': True, 'answer_path': 'batched_model_answer/llama3_1_8b_fp8_guided', 'bench_name': 'arena-hard-v0.1', 'guidance_only': True, 'max_tokens': 4096, 'model_list': ['llama3_1_8b_fp8_guided_run_1'], 'name': 'llama3_1_8b_fp8_guided-arena-hard-v0.1-4gpus-run_1', 'num_choices': 1, 'temperature': 0.0}
Guidance only mode
=========================  Expected Costs (based on GPT-4o)  =========================

Expected Input Tokens: 
 47461 Tokens in a total of 500 questions

Expected Output Tokens: 
 275000 Tokens in a total of 500 questions

Max Output Tokens: 
 400000 Tokens in a total of 500 questions


-------------------------  Resulting in Costs:   -------------------------

Expected Costs: 
 4.36 USD

Max. Expected Costs: 
 6.24 USD

Starting to generate answers...
Output to data/arena-hard-v0.1/batched_model_answer/llama3_1_8b_fp8_guided/llama3_1_8b_fp8_guided_run_1_guided.jsonl


 26%|██▌       | 128/500 [01:58<07:44,  1.25s/it]

ref: /opt/conda/envs/pytorch/lib/python3.11/site-packages/codecarbon/data/private_infra/2016/usa_emissions.json


 55%|█████▍    | 273/500 [03:57<02:22,  1.59it/s]

...

ref: /opt/conda/envs/pytorch/lib/python3.11/site-packages/codecarbon/data/private_infra/2016/usa_emissions.json
ref: /opt/conda/envs/pytorch/lib/python3.11/site-packages/codecarbon/data/private_infra/2016/usa_emissions.json
ref: /opt/conda/envs/pytorch/lib/python3.11/site-packages/codecarbon/data/private_infra/2016/usa_emissions.json
ref: /opt/conda/envs/pytorch/lib/python3.11/site-packages/codecarbon/data/private_infra/2016/usa_emissions.json
ref: /opt/conda/envs/pytorch/lib/python3.11/site-packages/codecarbon/data/private_infra/2016/usa_emissions.json
ref: /opt/conda/envs/pytorch/lib/python3.11/site-packages/codecarbon/data/private_infra/2016/usa_emissions.json
ref: /opt/conda/envs/pytorch/lib/python3.11/site-packages/codecarbon/data/private_infra/2016/usa_emissions.json
ref: /opt/conda/envs/pytorch/lib/python3.11/site-packages/codecarbon/data/private_infra/2016/usa_emissions.json
ref: /opt/conda/envs/pytorch/lib/python3.11/site-packages/codecarbon/data/private_infra/2016/usa_emissio

InternalServerError: Internal Server Error

ref: /opt/conda/envs/pytorch/lib/python3.11/site-packages/codecarbon/data/private_infra/2016/usa_emissions.json


Finished Benchmark in 3182.92s
=============== RESULTS for llama3_1_8b_fp8_guided-arena-hard-v0.1-4gpus-run_1 ===============


        Finished Benchmark arena-hard-v0.1 with llama3_1_8b_fp8


        Total Time: 3182.92s, AVG/Prompt: 6.37s


        Average tokens per second: 71.23


        Total Prompts: 500

        Total Input Tokens: 323272, AVG/Prompt: 646.544

        Total Output Tokens: 226727, AVG/Prompt: 453.454

        --------------------------------------------------



AVG. Input Tokens,▁
AVG. Output Tokens,▁
AVG. Time / Prompt,▁
AVG. Tokens / Second,▁
Emissions / 1.000.000 Input Tokens,▁
Emissions / 1.000.000 Output Tokens,▁
Emissions / 10.000 Prompts,▁
Total Emissions,▁
Total Time,▁
AVG. Input Tokens,646.544
AVG. Output Tokens,453.454


Results saved to emission_data/llama3_1_8b_fp8_guided-arena-hard-v0.1-4gpus-run_1_emission_data.csv


